In [5]:
import numpy as np
import pandas as pd

from pid.utils.generate import generate_cov_from_config, random_rotation_mxy, merge_covs

In [6]:
# Starting from 2-D case
dm, dx, dy = 2, 2, 2
theta = 0
gain_y = np.sqrt(2)
gains = np.linspace(0, 3, 10)
gains[3] = 0.99  # A gain of 1 is unstable

In [7]:
for i, gain_x in enumerate(gains):
    cov, *dims = generate_cov_from_config(d=(dm, dx, dy), gain=(gain_x, gain_y), theta=theta)
    # Generate random samples from the multivariate Gaussian distribution
    num_samples = 10  # Number of samples to generate
    mean = np.zeros(sum(dims))
    samples = np.random.multivariate_normal(mean, cov, num_samples)
    print(f'{samples} \n')

[[ 1.46414402  1.12457548  0.91841656 -1.41028379  2.13409889  1.25212971]
 [-1.59428479  0.46683726  1.75377777  0.22619063 -2.77961399 -0.04405417]
 [ 0.30209012 -2.17092023 -0.7015365  -1.44728705 -0.05678599 -2.11235887]
 [-0.20581202 -1.32703351  0.15161649 -0.62508109 -1.50420033 -1.06474339]
 [-0.94520081  2.55874031 -0.07328224  2.75449916 -2.26014575  5.2287524 ]
 [ 1.3114579   1.28007884 -0.44992274  1.3366322   0.96322421  1.68046279]
 [-0.52136545 -0.67984526 -0.73093095  0.47729033  1.63761302 -1.65468467]
 [-1.39408256  0.16795635  1.81744079  1.65161259 -1.71515584 -0.80757932]
 [ 0.95900255 -0.9735199   0.79184716 -0.43738396  1.57621358 -0.55991166]
 [-1.42414056 -0.34657128 -0.76986172 -0.16269935 -0.6556016  -0.88656783]] 

[[-0.89782534 -0.46802025 -0.63440233  1.43344512 -0.67577947 -0.84143992]
 [-0.6973757  -0.02899723 -1.34476906 -0.17740737 -0.78625561  0.55545005]
 [ 0.05650194 -0.3807265   0.39711871 -1.42305129  0.25296604 -0.0921957 ]
 [ 0.46074767  0.08620

In [11]:
# Extend to multi-dimensional case
gains = np.linspace(0, 3, 10)
gains[3] = 0.99  # A gain of 1 is unstable
#gains = np.array([3.0,])
num_doubles = 3
random_rotn = True

# for simplicity, I only print gains[0]
gain = gains[0]
for j in range(num_doubles):
    if j == 0:
        dm, dx, dy = 2, 2, 2
        hx = np.array([[gain, 0], [0, 1]])
        hy = np.array([[1, 0], [0, np.sqrt(2)]])
        sigm = np.eye(dm)
        sigx_m = np.eye(dx)
        sigy_m = np.eye(dy)
        sigw = np.zeros((dx, dy))
        cov = np.block([[sigm, sigm @ hx.T, sigm @ hy.T],
                        [hx @ sigm, hx @ sigm @ hx.T + sigx_m, hx @ sigm @ hy.T + sigw],
                        [hy @ sigm, hy @ sigm @ hx.T + sigw.T, hy @ sigm @ hy.T + sigy_m]])
        cov_old = cov.copy()
    else:
        cov, dm, dx, dy = merge_covs(cov_old, cov_old.copy(), dm, dx, dy)
        cov_old = cov.copy()

    if random_rotn:
        cov = random_rotation_mxy(cov, dm, dx, dy)

    num_samples = 5  # Number of samples to generate
    mean = np.zeros(sum([dm, dx, dy]))
    samples = np.random.multivariate_normal(mean, cov, num_samples)
    print(f'j: {j}, samples.shape: {samples.shape} \n {samples} \n ')

    dm = dx = dy = 2**(j+1)

j: 0, samples.shape: (5, 6) 
 [[ 0.72720771  0.65089014 -0.44257751 -0.0729151   2.20543715  2.74384103]
 [ 0.18786179 -0.90330147  0.70991029 -0.73690879  0.44671147 -1.75988621]
 [-0.8555556  -0.96364701  1.05819242 -1.03296247  0.03954473 -0.87584439]
 [-0.59617056  0.37498643  1.87880294 -0.0489639  -0.42464211 -0.46959098]
 [-0.50187478 -0.70103505  1.14995175  0.67802018  0.26266563 -2.17961459]] 
 
j: 1, samples.shape: (5, 12) 
 [[-1.18425093 -0.4132553  -0.55830811  0.19854681  1.47342386 -1.26327709
   0.62587798 -0.35233436 -0.28028658  0.05431966 -0.14700179 -1.1158658 ]
 [ 0.30038778 -0.63220527 -0.28520372  0.58232931  0.02765264  0.00504417
  -0.66560003 -0.07346881 -1.53148324 -1.12196074 -1.13004159 -1.43259511]
 [-1.80470911 -1.50708011  0.85455288 -0.60841342  0.7628887   0.90139331
   0.79326363  0.6059132  -0.71709483 -1.59175465  0.58453214 -2.65009844]
 [ 0.82013295 -0.05608554 -1.20701696  0.81318048  0.54122135  1.32592827
  -2.75696186  0.58352557  0.29296074 -